In [5]:
from pathlib import Path
import pandas as pd
DATA_PATH = Path("../data/malicious_phish.csv")
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head())
print(df.dtypes)

Shape: (651191, 2)
Columns: ['url', 'type']
                                                 url        type
0                                   br-icloud.com.br    phishing
1                mp3raid.com/music/krizz_kaliko.html      benign
2                    bopsecrets.org/rexroth/cr/1.htm      benign
3  http://www.garage-pirenne.be/index.php?option=...  defacement
4  http://adventure-nicaragua.net/index.php?optio...  defacement
url     str
type    str
dtype: object


In [6]:
print(df["type"].value_counts())
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nUnique labels:", df["type"].unique())

type
benign        428103
defacement     96457
phishing       94111
malware        32520
Name: count, dtype: int64

Missing values:
url     0
type    0
dtype: int64

Duplicate rows: 10066

Unique labels: <StringArray>
['phishing', 'benign', 'defacement', 'malware']
Length: 4, dtype: str


In [7]:
df = df.dropna(subset=["url", "type"]).copy()
df["url"] = df["url"].astype(str).str.strip()
df["type"] = df["type"].astype(str).str.strip().str.lower()
df = df[df["url"] != ""]
df = df.drop_duplicates(subset=["url", "type"]).reset_index(drop=True)

label_counts = df["type"].value_counts()
label_conflicts = df.groupby("url")["type"].nunique()
conflicting_urls = label_conflicts[label_conflicts > 1]

print("Cleaned shape:", df.shape)
print("\nClass distribution:")
print(label_counts)
print("\nURLs with conflicting labels:", len(conflicting_urls))
print("\nDuplicate URLs remaining:", df["url"].duplicated().sum())

Cleaned shape: (641125, 2)

Class distribution:
type
benign        428080
defacement     95308
phishing       94092
malware        23645
Name: count, dtype: int64

URLs with conflicting labels: 6

Duplicate URLs remaining: 6


In [8]:
from sklearn.model_selection import train_test_split

conflicting_urls = df.groupby("url")["type"].nunique()
conflicting_urls = conflicting_urls[conflicting_urls > 1].index

df = df[~df["url"].isin(conflicting_urls)].reset_index(drop=True)

X = df["url"]
y = df["type"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape[0])
print("Validation:", X_val.shape[0])
print("Testing:", X_test.shape[0])
print("\nTraining label distribution:")
print(y_train.value_counts())

Training: 512890
Validation: 64111
Testing: 64112

Training label distribution:
type
benign        342459
defacement     76246
phishing       75269
malware        18916
Name: count, dtype: int64


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report

tfidf_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char",
        ngram_range=(3, 5),
        max_features=100000,
        sublinear_tf=True,
        dtype="float32"
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

tfidf_model.fit(X_train, y_train)

val_predictions = tfidf_model.predict(X_val)

print("Validation accuracy:", accuracy_score(y_val, val_predictions))
print("Validation macro F1:", f1_score(y_val, val_predictions, average="macro"))
print("\nClassification report:")
print(classification_report(y_val, val_predictions, digits=4))

/opt/anaconda3/envs/phishingml/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:2049: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


Validation accuracy: 0.9705042816365367
Validation macro F1: 0.961315108912688

Classification report:
              precision    recall  f1-score   support

      benign     0.9897    0.9714    0.9804     42807
  defacement     0.9891    0.9978    0.9934      9531
     malware     0.9717    0.9586    0.9651      2365
    phishing     0.8733    0.9420    0.9063      9408

    accuracy                         0.9705     64111
   macro avg     0.9559    0.9674    0.9613     64111
weighted avg     0.9718    0.9705    0.9709     64111



In [10]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

test_predictions = tfidf_model.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, test_predictions))
print("Test macro F1:", f1_score(y_test, test_predictions, average="macro"))
print(classification_report(y_test, test_predictions, digits=4))
print("Confusion matrix:")
print(confusion_matrix(y_test, test_predictions, labels=tfidf_model.classes_))

Test accuracy: 0.9707699026703269
Test macro F1: 0.9606493498722579
              precision    recall  f1-score   support

      benign     0.9889    0.9729    0.9808     42808
  defacement     0.9904    0.9973    0.9938      9531
     malware     0.9667    0.9564    0.9615      2364
    phishing     0.8771    0.9377    0.9064      9409

    accuracy                         0.9708     64112
   macro avg     0.9558    0.9661    0.9606     64112
weighted avg     0.9719    0.9708    0.9711     64112

Confusion matrix:
[[41649    14    16  1129]
 [    5  9505     8    13]
 [    9     0  2261    94]
 [  454    78    54  8823]]


In [11]:
from pathlib import Path
import joblib

model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "tfidf_url_classifier.pkl"
joblib.dump(tfidf_model, model_path, compress=3)

print(f"Model saved to: {model_path.resolve()}")
print(f"Model size: {model_path.stat().st_size / (1024 * 1024):.2f} MB")

Model saved to: /Users/sreevatsarajesh/Documents/GitHub/FML_project/NLP_classifier/models/tfidf_url_classifier.pkl
Model size: 2.32 MB


In [12]:
import joblib
from pathlib import Path

model_path = Path("../models/tfidf_url_classifier.pkl")
nlp_model = joblib.load(model_path)

print("Model loaded:", type(nlp_model).__name__)
print("Classes:", nlp_model.classes_)
print("Pipeline steps:", list(nlp_model.named_steps.keys()))

Model loaded: Pipeline
Classes: ['benign' 'defacement' 'malware' 'phishing']
Pipeline steps: ['tfidf', 'classifier']
